## Create and Destroy a Storage Account & Container in Azure via Terraform

- An Azure Storage Account is used to store various types of data in Azure.
  - Since an Azure Storage Account is an Azure resource, it must be placed in an Azure Resource Group.
- A Storage Container is a specific storage type taht can be created under an Azure Storage Account.
  - A Storage Container is used to store BLOBs (Binary Large OBjects), i.e. an arbitrarty file (e.g. video file).
- This Terraform Project consists of the Terraform files listed below:

In [1]:
!dir *.tf
#!ls *.tf # use this on Linux/Mac

 Volume in drive C is Windows
 Volume Serial Number is 3C6C-8E33

 Directory of c:\Users\PAGA\projects\devops\workshop5\01_Azure_and_Terraform\05_storage_account

01/28/2025  06:42               655 container-registry.tf
01/27/2025  17:47               321 providers.tf
01/27/2025  17:47               152 resource-group.tf
01/28/2025  06:45             1,309 storage-account.tf
01/28/2025  06:48               553 storage-container.tf
01/27/2025  17:48               268 variables.tf
               6 File(s)          3,258 bytes
               0 Dir(s)  50,279,870,464 bytes free


## Terraform providers

- We are using the same Terraform proviers as before (i.e. the `azurerm` provider for Azure).

In [2]:
!type providers.tf
#!cat providers.tf # use this on Linux/Mac

# Initialises Terraform providers and sets their version numbers.

terraform {
  required_providers {
    azurerm = {
      source  = "hashicorp/azurerm"
      version = "~> 4.14.0"
    }
  }

  required_version = ">= 1.10.3"
}

provider "azurerm" {
  features {}
  subscription_id = var.subscription_id
}


## Terraform variables

- We are using the same Terraform variables as before (no variable for a kubernetes cluster this time).

**Note! Make sure you change the value for the variable `app_name` to something unique and set your `subscription_id`!**

In [4]:
!type variables.tf
#!cat variables.tf # use this on Linux/Mac

# Sets global variables for this Terraform project.

variable "subscription_id" {
  description = "The Azure subscription ID"
  type        = string
}

variable "app_name" {
  default = "flixtube2025g00"
}

variable "location" {
  default = "westeurope"
}


## Azure Resource Group

- We are using the same Azure Resoure Group as before.

In [5]:
!type resource-group.tf
#!cat resource-group.tf # use this on Linux/Mac

# Creates a resource group in your Azure account.

resource "azurerm_resource_group" "main" {
  name     = var.app_name
  location = var.location
}


## Azure Container Registry

- We are using the same Azure Container Registry as before.

In [6]:
!type container-registry.tf
#!cat container-registry.tf # use this on Linux/Mac

# Creates a container registry in Azure (for Docker images).

resource "azurerm_container_registry" "main" {
  name                = var.app_name
  resource_group_name = azurerm_resource_group.main.name
  location            = var.location
  admin_enabled       = true
  sku                 = "Basic"
}

output "AZURE_CONTAINER_REGISTRY_HOSTNAME" {
  value = azurerm_container_registry.main.login_server
}

output "AZURE_CONTAINER_REGISTRY_USERNAME" {
  value = azurerm_container_registry.main.admin_username
}

output "AZURE_CONTAINER_REGISTRY_PASSWORD" {
  value = azurerm_container_registry.main.admin_password
  sensitive = true
}


## Let's view the contents of the file `storage-account.tf`

- Here we are defining an Azure Storage Account
  - The Block Type is `resource`.
  - The first Block Label is `azurerm_storage_account`
    - `azurerm` is the name of the provider (i.e. the provider for Azure defined in the file `providers.tf`).
    - `storage_account` is the name of the Azure resource (i.e. an Azure Storage Account defined in the `azurerm` provider/plugin).
  - The second Block Label is `main`, which is used to uniquely identify the resource in the Terraform files.
  - The first Argument sets the Azure Storage Account's name
    - `name` is the argument's name
    - Its value is retrieved from the Terraform variable `app_name` (defined in the file `variables.tf`).
  - The second Argument sets the Azure Resource Group in which the Azure Storage Account will be created
    - `resource_group_name` is the argument's name
    - Its value is retrieved from the Terraform Expression `azurerm_resource_group.main.name`.
      - The `azurerm_resource_group.main` Block is defined in `resource-group.tf` as `resource "azurerm_resource_group" "main"`.
      - In this Block, there is an Argument with a name of `name` who's value is defined as `var.app_name`.
      - This is the value that is assigned to `resource_group_name`.
  - The third Argument sets the Azure Storage Account's Location
    - `location` is the argument's name
    - Its value is retrieved from the Terraform variable `location` (defined in the file `variables.tf`).
  - The fourth Argument sets the Storage Account's type of storage
    - `account_kind` is the argument's name
    - Its value is set to `"StorageV2"` (which is what we want if we want to add a Storage Container to the Storage Account).
    - For more information about the Azure Storage Account's account type, see:
      - https://registry.terraform.io/providers/hashicorp/azurerm/latest/docs/resources/storage_account
      - https://learn.microsoft.com/en-us/azure/storage/common/storage-account-overview
  - The fifth Argument sets the Storage Account's tier
    - `account_tier` is the argument's name
    - Its value is set to `"Standard"` (i.e. the Standard Tier which is a cheaper alternative).
    - For more information about the Azure Storage Account's tier, see:
      - https://registry.terraform.io/providers/hashicorp/azurerm/latest/docs/resources/storage_account
      - https://learn.microsoft.com/en-us/azure/storage/common/storage-account-overview
  - The fifth Argument sets the Storage Account's replication type.
    - `account_replication_type` is the argument's name
    - Its value is set to `"LRS"` (for Local Redundant Storage).
    - For more information about the Azure Storage Account's replication type, see:
      - https://registry.terraform.io/providers/hashicorp/azurerm/latest/docs/resources/storage_account
      - https://learn.microsoft.com/en-us/azure/storage/common/storage-account-overview
  - The ouputs are used to print out the Azure Storage Account's name and key.
    - The key is sensitive and will be redacted when printed out.

In [7]:
!type storage-account.tf
#!cat storage-account.tf # use this on Linux/Mac

# Create a storage account in Azure

# Note!
# - Resource "azurerm_resource_group.main" with a property "name" is defined in the file "resource-group.tf".
# - The value for "resource_group_name" below is set using property "name" in resource "azurerm_resource_group.main":
#   - resource_group_name = azurerm_resource_group.main.name
# - "name" and "location" below are set from Terraform variables defined in the file "variables.tf".
# - The value for "storage_account_name" below is set using property "name" in resource "azurerm_storage_account.main":
#   - storage_account_name = azurerm_storage_account.main.name
# - The ouputs below are used to print out the Azure Storage Account's name and key.
#   - The key is sensitive and will be redacted when printed out.

resource "azurerm_storage_account" "main" {
  name                     = var.app_name
  resource_group_name      = azurerm_resource_group.main.name
  location                 = var.location
  # account_kind             = "StorageV

## Let's view the contents of the file `storage-container.tf`

- Here we are defining an Azure Storage Container (which is used to store data as BLOBs, i.e. arbitrary files)
  - The Block Type is `resource`.
  - The first Block Label is `azurerm_storage_container`
    - `azurerm` is the name of the provider (i.e. the provider for Azure defined in the file `providers.tf`).
    - `storage_container` is the name of the Azure resource (i.e. an Azure Storage Container defined in the `azurerm` provider/plugin).
  - The second Block Label is `main`, which is used to uniquely identify the resouce in the Terraform files.
  - The first Argument sets the Storage Container's name
    - `name` is the argument's name
    - Its value is set to `videos` (this is equivalent to the name of a folder, which will contain files).
  - The second Argument sets the Azure Storage Account in which the Storage Container will be created
    - `storage_account_name` is the argument's name
    - Its value is retrieved from the Terraform Expression `azurerm_storage_account.main.name`.
      - The `azurerm_storage_account.main` Block is defined in `storage-account.tf`
        - as `resource "azurerm_storage_account" "main"`.
      - In this Block, there is an Argument with a name of `name` who's value is defined as `var.app_name`.
      - This is the value that is assigned to `storage_account_name`.
  - The third Argument sets the Storage Container's access type
    - `container_access_type` is the argument's name
    - Its value is set to `private` (which means public access to the Storage Container is disabled).
    - For more information about the Azure Storage Container's acces type, see:
      - https://registry.terraform.io/providers/hashicorp/azurerm/latest/docs/resources/storage_container
      - https://learn.microsoft.com/en-us/azure/private-link/tutorial-private-endpoint-storage-portal?tabs=dynamic-ip

In [8]:
!type storage-container.tf
#!cat storage-container.tf # use this on Linux/Mac

# Create a storage container in Azure (for BLoBs, i.e. arbitrary files)

# Note!
# - Resource "azurerm_storage_account.main" with a property "id" is defined in the file "storage-account.tf".
# - The value for "storage_account_id" below is set using property "id" in resource "azurerm_storage_account.main":
#   - storage_account_id  = azurerm_storage_account.main.id

resource "azurerm_storage_container" "main" {
  name                  = "videos"
  storage_account_id  = azurerm_storage_account.main.id
  container_access_type = "private"
}


## Initialize Terraform

- We initialize the Terraform Project as before.

In [9]:
#rm -rf .terraform rm .terraform.lock.hcl terraform.tfstate terraform.tfstate.backup
!terraform init

Initializing the backend...
Initializing provider plugins...
- Finding hashicorp/azurerm versions matching "~> 4.14.0"...
- Installing hashicorp/azurerm v4.14.0...
- Installed hashicorp/azurerm v4.14.0 (signed by HashiCorp)
Terraform has created a lock file .terraform.lock.hcl to record the provider
selections it made above. Include this file in your version control repository
so that Terraform can guarantee to make the same selections by default when
you run "terraform init" in the future.

Terraform has been successfully initialized!

You may now begin working with Terraform. Try running "terraform plan" to see
any changes that are required for your infrastructure. All Terraform commands
should now work.

If you ever set or change modules or backend configuration for Terraform,
rerun this command to reinitialize your working directory. If you forget, other
commands will detect it and remind you to do so if necessary.


## Terraform Apply

- We apply the Terraform Project as before.

**Note!**

- Once again it's better to run this command in a separate terminal.
  - Open a new terminal.
  - Make sure you are in the folder `workshop5/01_Azure_and_Terraform/05_storage_account`
  - Execute the command `terraform apply -auto-approve`

The output should look something like the below ...

```bash
Terraform used the selected providers to generate the following execution plan. Resource actions are indicated with the following symbols:
  + create

Terraform will perform the following actions:

  # azurerm_container_registry.main will be created
  + resource "azurerm_container_registry" "main" {
      + admin_enabled                 = true
      + admin_password                = (sensitive value)
      + admin_username                = (known after apply)
      + encryption                    = (known after apply)
      + export_policy_enabled         = true
      + id                            = (known after apply)
      + location                      = "westeurope"
      + login_server                  = (known after apply)
      + name                          = "flixtube2025g00"
      + network_rule_bypass_option    = "AzureServices"
      + network_rule_set              = (known after apply)
      + public_network_access_enabled = true
      + resource_group_name           = "flixtube2025g00"
      + sku                           = "Basic"
      + trust_policy_enabled          = false
      + zone_redundancy_enabled       = false
    }

  # azurerm_resource_group.main will be created
  + resource "azurerm_resource_group" "main" {
      + id       = (known after apply)
      + location = "westeurope"
      + name     = "flixtube2025g00"
    }

  # azurerm_storage_account.main will be created
  + resource "azurerm_storage_account" "main" {
      + access_tier                        = (known after apply)
      + account_kind                       = "StorageV2"
      + account_replication_type           = "LRS"
      + account_tier                       = "Standard"
      + allow_nested_items_to_be_public    = true
      + cross_tenant_replication_enabled   = false
      + default_to_oauth_authentication    = false
      + dns_endpoint_type                  = "Standard"
      + https_traffic_only_enabled         = true
      + id                                 = (known after apply)
      + infrastructure_encryption_enabled  = false
      + is_hns_enabled                     = false
      + large_file_share_enabled           = (known after apply)
      + local_user_enabled                 = true
      + location                           = "westeurope"
      + min_tls_version                    = "TLS1_2"
      + name                               = "flixtube2025g00"
      + nfsv3_enabled                      = false
      + primary_access_key                 = (sensitive value)
      + primary_blob_connection_string     = (sensitive value)
      + primary_blob_endpoint              = (known after apply)
      + primary_blob_host                  = (known after apply)
      + primary_blob_internet_endpoint     = (known after apply)
      + primary_blob_internet_host         = (known after apply)
      + primary_blob_microsoft_endpoint    = (known after apply)
      + primary_blob_microsoft_host        = (known after apply)
      + primary_connection_string          = (sensitive value)
      + primary_dfs_endpoint               = (known after apply)
      + primary_dfs_host                   = (known after apply)
      + primary_dfs_internet_endpoint      = (known after apply)
      + primary_dfs_internet_host          = (known after apply)
      + primary_dfs_microsoft_endpoint     = (known after apply)
      + primary_dfs_microsoft_host         = (known after apply)
      + primary_file_endpoint              = (known after apply)
      + primary_file_host                  = (known after apply)
      + primary_file_internet_endpoint     = (known after apply)
      + primary_file_internet_host         = (known after apply)
      + primary_file_microsoft_endpoint    = (known after apply)
      + primary_file_microsoft_host        = (known after apply)
      + primary_location                   = (known after apply)
      + primary_queue_endpoint             = (known after apply)
      + primary_queue_host                 = (known after apply)
      + primary_queue_microsoft_endpoint   = (known after apply)
      + primary_queue_microsoft_host       = (known after apply)
      + primary_table_endpoint             = (known after apply)
      + primary_table_host                 = (known after apply)
      + primary_table_microsoft_endpoint   = (known after apply)
      + primary_table_microsoft_host       = (known after apply)
      + primary_web_endpoint               = (known after apply)
      + primary_web_host                   = (known after apply)
      + primary_web_internet_endpoint      = (known after apply)
      + primary_web_internet_host          = (known after apply)
      + primary_web_microsoft_endpoint     = (known after apply)
      + primary_web_microsoft_host         = (known after apply)
      + public_network_access_enabled      = true
      + queue_encryption_key_type          = "Service"
      + resource_group_name                = "flixtube2025g00"
      + secondary_access_key               = (sensitive value)
      + secondary_blob_connection_string   = (sensitive value)
      + secondary_blob_endpoint            = (known after apply)
      + secondary_blob_host                = (known after apply)
      + secondary_blob_internet_endpoint   = (known after apply)
      + secondary_blob_internet_host       = (known after apply)
      + secondary_blob_microsoft_endpoint  = (known after apply)
      + secondary_blob_microsoft_host      = (known after apply)
      + secondary_connection_string        = (sensitive value)
      + secondary_dfs_endpoint             = (known after apply)
      + secondary_dfs_host                 = (known after apply)
      + secondary_dfs_internet_endpoint    = (known after apply)
      + secondary_dfs_internet_host        = (known after apply)
      + secondary_dfs_microsoft_endpoint   = (known after apply)
      + secondary_dfs_microsoft_host       = (known after apply)
      + secondary_file_endpoint            = (known after apply)
      + secondary_file_host                = (known after apply)
      + secondary_file_internet_endpoint   = (known after apply)
      + secondary_file_internet_host       = (known after apply)
      + secondary_file_microsoft_endpoint  = (known after apply)
      + secondary_file_microsoft_host      = (known after apply)
      + secondary_location                 = (known after apply)
      + secondary_queue_endpoint           = (known after apply)
      + secondary_queue_host               = (known after apply)
      + secondary_queue_microsoft_endpoint = (known after apply)
      + secondary_queue_microsoft_host     = (known after apply)
      + secondary_table_endpoint           = (known after apply)
      + secondary_table_host               = (known after apply)
      + secondary_table_microsoft_endpoint = (known after apply)
      + secondary_table_microsoft_host     = (known after apply)
      + secondary_web_endpoint             = (known after apply)
      + secondary_web_host                 = (known after apply)
      + secondary_web_internet_endpoint    = (known after apply)
      + secondary_web_internet_host        = (known after apply)
      + secondary_web_microsoft_endpoint   = (known after apply)
      + secondary_web_microsoft_host       = (known after apply)
      + sftp_enabled                       = false
      + shared_access_key_enabled          = true
      + table_encryption_key_type          = "Service"

      + blob_properties (known after apply)

      + network_rules (known after apply)

      + queue_properties (known after apply)

      + routing (known after apply)

      + share_properties (known after apply)

      + static_website (known after apply)
    }

  # azurerm_storage_container.main will be created
  + resource "azurerm_storage_container" "main" {
      + container_access_type             = "private"
      + default_encryption_scope          = (known after apply)
      + encryption_scope_override_enabled = true
      + has_immutability_policy           = (known after apply)
      + has_legal_hold                    = (known after apply)
      + id                                = (known after apply)
      + metadata                          = (known after apply)
      + name                              = "videos"
      + resource_manager_id               = (known after apply)
      + storage_account_id                = (known after apply)
    }

Plan: 4 to add, 0 to change, 0 to destroy.

Changes to Outputs:
  + AZURE_CONTAINER_REGISTRY_HOSTNAME = (known after apply)
  + AZURE_CONTAINER_REGISTRY_PASSWORD = (sensitive value)
  + AZURE_CONTAINER_REGISTRY_USERNAME = (known after apply)
  + AZURE_STORAGE_ACCOUNT_KEY         = (sensitive value)
  + AZURE_STORAGE_ACCOUNT_NAME        = "flixtube2025g00"
azurerm_resource_group.main: Creating...
azurerm_resource_group.main: Still creating... [10s elapsed]
azurerm_resource_group.main: Creation complete after 12s [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00]
azurerm_container_registry.main: Creating...
azurerm_storage_account.main: Creating...
azurerm_container_registry.main: Still creating... [10s elapsed]
azurerm_storage_account.main: Still creating... [10s elapsed]
azurerm_container_registry.main: Still creating... [20s elapsed]
azurerm_storage_account.main: Still creating... [20s elapsed]
azurerm_container_registry.main: Creation complete after 24s [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00]
azurerm_storage_account.main: Still creating... [30s elapsed]
azurerm_storage_account.main: Still creating... [40s elapsed]
azurerm_storage_account.main: Still creating... [50s elapsed]
azurerm_storage_account.main: Still creating... [1m0s elapsed]
azurerm_storage_account.main: Creation complete after 1m7s [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.Storage/storageAccounts/flixtube2025g00]
azurerm_storage_container.main: Creating...
azurerm_storage_container.main: Creation complete after 2s [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.Storage/storageAccounts/flixtube2025g00/blobServices/default/containers/videos]

Apply complete! Resources: 4 added, 0 changed, 0 destroyed.

Outputs:

AZURE_CONTAINER_REGISTRY_HOSTNAME = "flixtube2025g00.azurecr.io"
AZURE_CONTAINER_REGISTRY_PASSWORD = <sensitive>
AZURE_CONTAINER_REGISTRY_USERNAME = "flixtube2025g00"
AZURE_STORAGE_ACCOUNT_KEY = <sensitive>
AZURE_STORAGE_ACCOUNT_NAME = "flixtube2025g00"
```

In [10]:
# !terraform apply -auto-approve

## List Azure Resource Groups

- The Azure CLI command `az group list -o table` lists all Resource Groups in Azure.
- We see that the Resource Group defined in the Terraform project has been created.

In [11]:
!az group list -o table

Name             Location    Status
---------------  ----------  ---------
flixtube2025g00  westeurope  Succeeded


## List Resources in Resource Group `flixtube2025g00`

- The Azure CLI command `az resource list -n flixtube2025g00 -o table` lists resources in Resource Group `flixtube2025g00`.
- We see that the Resource Group contains the Storage Account and the Container Registry.

In [12]:
!az resource list -n flixtube2025g00 -o table

Name             ResourceGroup    Location    Type                                    Status
---------------  ---------------  ----------  --------------------------------------  --------
flixtube2025g00  flixtube2025g00  westeurope  Microsoft.Storage/storageAccounts
flixtube2025g00  flixtube2025g00  westeurope  Microsoft.ContainerRegistry/registries


## List Azure Storage Accounts

- The Azure CLI command `az storage account list -o table` lists all Storage Accounts in Azure.
- We see that the Storage Account defined in the Terraform project has been cerated.

In [13]:
#!az storage account list --resource-group $RESOURCE_GROUP -o table
!az storage account list -o table

AccessTier    AllowBlobPublicAccess    AllowCrossTenantReplication    AllowSharedKeyAccess    CreationTime                      DefaultToOAuthAuthentication    DnsEndpointType    EnableHttpsTrafficOnly    EnableNfsV3    IsHnsEnabled    IsLocalUserEnabled    IsSftpEnabled    Kind       Location    MinimumTlsVersion    Name             PrimaryLocation    ProvisioningState    PublicNetworkAccess    ResourceGroup    StatusOfPrimary
------------  -----------------------  -----------------------------  ----------------------  --------------------------------  ------------------------------  -----------------  ------------------------  -------------  --------------  --------------------  ---------------  ---------  ----------  -------------------  ---------------  -----------------  -------------------  ---------------------  ---------------  -----------------
Hot           True                     False                          True                    2025-01-28T07:05:09.082419+00:00  False 

## List Storage Account Access Keys for Storage Account `flixtube2025g00`

- The Azure CLI command `az storage account keys list --account-name flixtube2025g00 -o table` will
  - List Access Keys for the Storage Account named `flixtube2025g00`.
- It shows the two keys `key1` and `key2` of which any can be used together with the Storage Account `name` to acccess the Storage Account.
  - **This is the KEY (key1 or key2) you would use together with the storage account NAME to upload/download/delete files.**

In [ ]:
#!az storage account keys list --account-name flixtube2025g00 --resource-group tsfn14g00 -o table
!az storage account keys list --account-name flixtube2025g00 -o table

# Let's store the STORAGE ACCOUNT NAME and STORAGE ACCES KEY in Python variables so we can use them later in this notebook
STORAGE_ACCOUNT_NAME=!az storage account list --query [0].name -o tsv
STORAGE_ACCOUNT_NAME=STORAGE_ACCOUNT_NAME[0]
STORAGE_ACCESS_KEY=!az storage account keys list --account-name flixtube2025g00 --resource-group flixtube2025g00 --query [0].value -o tsv
STORAGE_ACCESS_KEY=STORAGE_ACCESS_KEY[0]

## List Containers in Storage Account `flixtube2025g00`

- The command `az storage container list --account-name $STORAGE_ACCOUNT_NAME --account-key $STORAGE_ACCESS_KEY -o table`
  - Lists the Storage Containers in Storage Account `$STORAGE_ACCOUNT_NAME` (using the Key `$STORAGE_ACCESS_KEY`).

In [15]:
!az storage container list --account-name $STORAGE_ACCOUNT_NAME --account-key $STORAGE_ACCESS_KEY -o table

Name    Lease Status    Last Modified
------  --------------  -------------------------
videos                  2025-01-28T07:06:15+00:00


## Create Container `videos2` in Storage Container `flixtube2025g00`

- The command:

  ```bash
  az storage container create --name videos2 --account-name $STORAGE_ACCOUNT_NAME --account-key $STORAGE_ACCESS_KEY -o table
  ```
  - Creates a new Storage COntainer named `videos2` in Storage Account `$STORAGE_ACCOUNT_NAME` (using KEY `$STORAGE_ACCESS_KEY`).

In [16]:
!az storage container create --name videos2 --account-name $STORAGE_ACCOUNT_NAME --account-key $STORAGE_ACCESS_KEY -o table

Created
---------
True


## List Containers in Storage Container `flixtube2025g00`

- Now when we list the containes in Storage Account `flixtube2025g00` again, we see there are two Storage Containers.

In [17]:
!az storage container list --account-name $STORAGE_ACCOUNT_NAME --account-key $STORAGE_ACCESS_KEY -o table

Name     Lease Status    Last Modified
-------  --------------  -------------------------
videos                   2025-01-28T07:06:15+00:00
videos2                  2025-01-28T07:12:06+00:00


## Delete Container `videos2` in Storage Container `flixtube2025g00`

- The command:

  ```bash
  az storage container delete --name videos2 --account-name $STORAGE_ACCOUNT_NAME --account-key $STORAGE_ACCESS_KEY -o table
  ```
  - Deletes the Storage COntainer named `videos2` from Storage Account `$STORAGE_ACCOUNT_NAME` (using KEY `$STORAGE_ACCESS_KEY`).

In [18]:
!az storage container delete --name videos2 --account-name $STORAGE_ACCOUNT_NAME --account-key $STORAGE_ACCESS_KEY -o table

Deleted
---------
True


## List Containers in Storage Container `flixtube2025g00`

- Now when we list the containes in Storage Account `flixtube2025g00` again, we see the Storage Container has been deleted.

In [19]:
!az storage container list --account-name $STORAGE_ACCOUNT_NAME --account-key $STORAGE_ACCESS_KEY -o table

Name    Lease Status    Last Modified
------  --------------  -------------------------
videos                  2025-01-28T07:06:15+00:00


## List Blobs (Files) in Container `videos` in Storage Container `flixtube2025g00`

- The command:

  ```bash
  az storage blob list --container-name videos --account-name $STORAGE_ACCOUNT_NAME --account-key $STORAGE_ACCESS_KEY
  ```
  - Lists files stored in the Storage Container named `videos` in Storage Account `$STORAGE_ACCOUNT_NAME` (using KEY `$STORAGE_ACCESS_KEY`).
  - Currently the lsit is empty, since no files are stored in Storage Container `videos`.

In [20]:
!az storage blob list --container-name videos --account-name $STORAGE_ACCOUNT_NAME --account-key $STORAGE_ACCESS_KEY -o table 

## List Local Video File

- There is a video file stored in the host machine's file system: `Flixtube.AzureStorage/videos/SampleVideo_1280x720_1mb.mp4`.

In [21]:
!dir Flixtube.AzureStorage\videos
#!ls Flixtube.AzureStorage/videos # use this on Linux/Mac

 Volume in drive C is Windows
 Volume Serial Number is 3C6C-8E33

 Directory of c:\Users\PAGA\projects\devops\workshop5\01_Azure_and_Terraform\05_storage_account\Flixtube.AzureStorage\videos

01/28/2025  08:13    <DIR>          .
01/28/2025  08:13    <DIR>          ..
12/22/2024  12:29         1,055,736 SampleVideo_1280x720_1mb.mp4
               1 File(s)      1,055,736 bytes
               2 Dir(s)  49,975,128,064 bytes free


## Upload a file to Container `videos` in Storage Container `flixtube2025g00`

- The command below:
  - Uploads the local file `Flixtube.AzureStorage/videos/SampleVideo_1280x720_1mb.mp4` (`--file`).
  - To the Storage Container `videos` (`--container-name`).
  - In Storage Account `$STORAGE_ACCOUNT_NAME` (`--account-name`).
  - using Key `$STORAGE_ACCESS_KEY` (`--account-key`).

In [ ]:
!az storage blob upload --container-name videos --name SampleVideo_1280x720_1mb.mp4 --file Flixtube.AzureStorage/videos/SampleVideo_1280x720_1mb.mp4 --account-name $STORAGE_ACCOUNT_NAME --account-key $STORAGE_ACCESS_KEY -o table

## List Blobs (Files) in Container `videos` in Storage Container `flixtube2025g00`

- Now if we list the files in Storage Container `videos` again, we see the video file has been uploaded.

In [23]:
!az storage blob list --container-name videos --account-name $STORAGE_ACCOUNT_NAME --account-key $STORAGE_ACCESS_KEY -o table 

Name                          Blob Type    Blob Tier    Length    Content Type    Last Modified              Snapshot
----------------------------  -----------  -----------  --------  --------------  -------------------------  ----------
SampleVideo_1280x720_1mb.mp4  BlockBlob    Hot          1055736   video/mp4       2025-01-28T07:16:14+00:00


## Delete Local Video File

- Let's delete the video file from the host machine.

In [25]:
!del Flixtube.AzureStorage\videos\SampleVideo_1280x720_1mb.mp4
!dir Flixtube.AzureStorage\videos

#!rm Flixtube.AzureStorage/videos/SampleVideo_1280x720_1mb.mp4 # use this on Linux/Mac
#!ls Flixtube.AzureStorage/videos # use this on Linux/Mac

 Volume in drive C is Windows
 Volume Serial Number is 3C6C-8E33

 Directory of c:\Users\PAGA\projects\devops\workshop5\01_Azure_and_Terraform\05_storage_account\Flixtube.AzureStorage\videos

01/28/2025  08:18    <DIR>          .
01/28/2025  08:18    <DIR>          ..
               0 File(s)              0 bytes
               2 Dir(s)  49,973,456,896 bytes free


## Download a file from Container `videos` in Storage Account `flixtube2025g00`

- The command below:
  - Dowloads the file `SampleVideo_1280x720_1mb.mp4` (`--name`).
  - From Storage Container `videos` (`--container-name`).
  - In Storage Account `$STORAGE_ACCOUNT_NAME` (`--account-name`).
  - To the local file `Flixtube.AzureStorage/videos/SampleVideo_1280x720_1mb.mp4` (`--file`).
  - using Key `$STORAGE_ACCESS_KEY` (`--account-key`).

In [26]:
!az storage blob download --container-name videos --name SampleVideo_1280x720_1mb.mp4 --file Flixtube.AzureStorage/videos/SampleVideo_1280x720_1mb.mp4 --account-name $STORAGE_ACCOUNT_NAME --account-key $STORAGE_ACCESS_KEY -o table

Name                          Blob Type    Blob Tier    Length    Content Type    Last Modified              Snapshot
----------------------------  -----------  -----------  --------  --------------  -------------------------  ----------
SampleVideo_1280x720_1mb.mp4  BlockBlob                 1055736   video/mp4       2025-01-28T07:16:14+00:00



Alive[################################################################]  100.0000%
Finished[#############################################################]  100.0000%


## List Local Video File

- We see that the file has been downlaoded to the host machine.

In [28]:
!dir Flixtube.AzureStorage\videos
#!ls Flixtube.AzureStorage/videos # use this on Linux/Mac

 Volume in drive C is Windows
 Volume Serial Number is 3C6C-8E33

 Directory of c:\Users\PAGA\projects\devops\workshop5\01_Azure_and_Terraform\05_storage_account\Flixtube.AzureStorage\videos

01/28/2025  08:19    <DIR>          .
01/28/2025  08:19    <DIR>          ..
01/28/2025  08:19         1,055,736 SampleVideo_1280x720_1mb.mp4
               1 File(s)      1,055,736 bytes
               2 Dir(s)  49,971,163,136 bytes free


## Delete a file from Container `videos` in Storage Container `flixtube2025g00`

- The command below:
  - Deletes the file `SampleVideo_1280x720_1mb.mp4` (`--name`).
  - From Storage Container `videos` (`--container-name`).
  - In Storage Account `$STORAGE_ACCOUNT_NAME` (`--account-name`).
  - using Key `$STORAGE_ACCESS_KEY` (`--account-key`).

In [29]:
!az storage blob delete --container-name videos --name SampleVideo_1280x720_1mb.mp4 --account-name $STORAGE_ACCOUNT_NAME --account-key $STORAGE_ACCESS_KEY -o table

## List Blobs (Files) in Container `videos` in Storage Container `flixtube2025g00`

- Now if we list the files in Storage Container `videos` again, we see the video file has been deleted.

In [30]:
!az storage blob list --container-name videos --account-name $STORAGE_ACCOUNT_NAME --account-key $STORAGE_ACCESS_KEY -o table 

## Upload file again to Container `videos` in Storage Container `flixtube2025g00`

- Let's upload the video file to the Storage Container again. 

In [31]:
!az storage blob upload --container-name videos --name SampleVideo_1280x720_1mb.mp4 --file Flixtube.AzureStorage/videos/SampleVideo_1280x720_1mb.mp4 --account-name $STORAGE_ACCOUNT_NAME --account-key $STORAGE_ACCESS_KEY -o table
!az storage blob list --container-name videos --account-name $STORAGE_ACCOUNT_NAME --account-key $STORAGE_ACCESS_KEY -o table 

Client_request_id                     Content_md5               Date                       LastModified               Request_id                            Request_server_encrypted    Version
------------------------------------  ------------------------  -------------------------  -------------------------  ------------------------------------  --------------------------  ----------
76c58f50-dd48-11ef-b598-38fc98105c26  1Vvd+NYpEIee2fYFUiFJqA==  2025-01-28T07:21:16+00:00  2025-01-28T07:21:17+00:00  5e2e59e5-901e-0044-0955-718398000000  True                        2022-11-02



Alive[################################################################]  100.0000%
Finished[#############################################################]  100.0000%


Name                          Blob Type    Blob Tier    Length    Content Type    Last Modified              Snapshot
----------------------------  -----------  -----------  --------  --------------  -------------------------  ----------
SampleVideo_1280x720_1mb.mp4  BlockBlob    Hot          1055736   video/mp4       2025-01-28T07:21:17+00:00


## List Azure Container Registries

- The Azure CLI command `az acr list -o table` lists all Container Registries in Azure.
- We see that the Container Registry defined in the Terraform project has been created.

In [32]:
!az acr list -o table

NAME             RESOURCE GROUP    LOCATION    SKU    LOGIN SERVER                CREATION DATE         ADMIN ENABLED
---------------  ----------------  ----------  -----  --------------------------  --------------------  ---------------
flixtube2025g00  flixtube2025g00   westeurope  Basic  flixtube2025g00.azurecr.io  2025-01-28T07:05:11Z  True


## List Repositories in Container Registry `flixtube2025g00 `

- The Azure CLI command `az acr repository list -n flixtube2025g00 --top 10 -o table` lists Repositories in a Azure Container Registry `flixtube2025g00`.
  - The `-n` option is manditory and specifies the `NAME` of the Container Registry.
  - The `--top 10` limits the list to the first 10 Repositories (remove to see all Repositories).
- We see that Container Registry `flixtube2025g00` doesn't contain any Repositories.

In [33]:
!az acr repository list -n flixtube2025g00 --top 10 -o table

## Show Information about Azure Container Registry `flixtube2025g00`

- The Azure CLI command `az acr show -n flixtube2025g00 -o table` shows information about Container Registry `flixtube2025g00`.
- It shows the Container Registry's `LOGIN SERVER` which is the URL to your Container Registry on Azure.
  - **This is the URL you would use to upload Docker Images to the Azure Container Registry.**

In [34]:
!az acr show -n flixtube2025g00 -o table

# Let's store the LOGIN SERVER in a Python variable so we can use it later in this notebook
CONTAINER_REGISTRY_LOGIN_SERVER=!az acr show -n flixtube2025g00 --query loginServer -o tsv
CONTAINER_REGISTRY_LOGIN_SERVER=CONTAINER_REGISTRY_LOGIN_SERVER[0]

NAME             RESOURCE GROUP    LOCATION    SKU    LOGIN SERVER                CREATION DATE         ADMIN ENABLED
---------------  ----------------  ----------  -----  --------------------------  --------------------  ---------------
flixtube2025g00  flixtube2025g00   westeurope  Basic  flixtube2025g00.azurecr.io  2025-01-28T07:05:11Z  True


## Show Credentials for Azure Container Registry `flixtube2025g00`

- The Azure CLI command `az acr credential show -n flixtube2025g00 -o table` shows credentials about Container Registry `flixtube2025g00`.
- It shows the Container Registry's `USERNAME` and  `PASSWORD` to use to authenticate with your Azure Container Registry.
  - **This is the USERNAME and PASSWORD you would use to login to Docker to upload Images to the Azure Container Registry.**

In [ ]:
!az acr credential show -n flixtube2025g00 -o table

# Let's store the USERNAME and PASSWORD in Python variables so we can use them later in this notebook
CONTAINER_REGISTRY_USERNAME=!az acr credential show -n flixtube2025g00 --query username -o tsv
CONTAINER_REGISTRY_USERNAME=CONTAINER_REGISTRY_USERNAME[0]
CONTAINER_REGISTRY_PASSWORD=!az acr credential show -n flixtube2025g00 --query passwords[0].value -o tsv
CONTAINER_REGISTRY_PASSWORD=CONTAINER_REGISTRY_PASSWORD[0]

## Login to Azure Container Registry via Docker

- Replace the variables below with your `LOGIN_SERVER`, `USERNAME` and `PASSWORD`.

In [36]:
!docker login $CONTAINER_REGISTRY_LOGIN_SERVER -u $CONTAINER_REGISTRY_USERNAME -p $CONTAINER_REGISTRY_PASSWORD

# In Ubuntu with environment variables CONTAINER_REGISTRY_LOGIN_SERVER, CONTAINER_REGISTRY_USERNAME and CONTAINER_REGISTRY_USERNAME
#!echo $CONTAINER_REGISTRY_PASSWORD | docker login $CONTAINER_REGISTRY_LOGIN_SERVER -u $CONTAINER_REGISTRY_USERNAME --password-stdin  > /dev/null 2>&1

Login Succeeded


WARNING! Using --password via the CLI is insecure. Use --password-stdin.


## Let's view the code in `Flixtube.AzureStorage`

- The file `Flixtube.AzureStorage/Flixtube.AzureStorage/Program.cs` is listed below.
  - At the top of the file, it reads in a four environment variables.
    - `FLIXTUBE_AZURE_STORAGE_PORT` is the port the microservice will listen on.
    - `FLIXTUBE_STORAGE_ACCOUNT_NAME` is the name of the storage account.
    - `FLIXTUBE_STORAGE_ACCESS_KEY` is the key to the storage account.
    - `FLIXTUBE_STORAGE_CONTAINER_NAME` is the name of the container in the storage account.
  - These environment variables are stored in the `IConfiguration` instance, after discarding their `FLIXTUBE_` prefix.
  - Then it configures the service container and the HTTP Request/Response pipeline as usual for an ASP.NET Web API project.
  - Lastly, it starts the microservice listening on port `FLIXTUBE_AZURE_STORAGE_PORT`.

In [41]:
!type Flixtube.AzureStorage\Flixtube.AzureStorage\Program.cs
#!cat Flixtube.AzureStorage/Flixtube.AzureStorage/Program.cs # use this on Linux/Mac

var builder = WebApplication.CreateBuilder(args);

// Make sure the necessary environment variables are available.
if (string.IsNullOrEmpty(Environment.GetEnvironmentVariable("FLIXTUBE_AZURE_STORAGE_PORT"))) {
    throw new Exception("Please specify the port number for Flixtube.AzureStorage with the environment variable FLIXTUBE_AZURE_STORAGE_PORT.");
}

if (string.IsNullOrEmpty(Environment.GetEnvironmentVariable("FLIXTUBE_STORAGE_ACCOUNT_NAME"))) {
    throw new Exception("Please specify the name of an Azure storage account in environment variable FLIXTUBE_STORAGE_ACCOUNT_NAME.");
}

if (string.IsNullOrEmpty(Environment.GetEnvironmentVariable("FLIXTUBE_STORAGE_ACCESS_KEY"))) {
    throw new Exception("Please specify the access key to an Azure storage account in environment variable FLIXTUBE_STORAGE_ACCESS_KEY.");
}

if (string.IsNullOrEmpty(Environment.GetEnvironmentVariable("FLIXTUBE_STORAGE_CONTAINER_NAME"))) {
    throw new Exception("Please specify the name of an Azure storage con

- The file `Flixtube.AzureStorage/Flixtube.AzureStorage/Controllers/AzureStorageController.cs` is listed below.
  - At the top of the file, it reads in three environment variable stored in the dependency-injected `IConfiguration` instance.
    - `STORAGE_ACCOUNT_NAME` is the name of the storage account.
    - `STORAGE_ACCESS_KEY` is the key to the storage account.
    - `STORAGE_CONTAINER_NAME` is the name of the container in the storage account.
  - The file also contains a number of HTTP endpoints, where one of these is the `/video` route that streams a video from the storage account.
    - It expects a query parameter `id` which is the name of the video file stored in the storage account.
    - `STORAGE_CONTAINER_NAME` is the name of the container that stores the video files (blobs).
    - `STORAGE_FOLDER_NAME/id` is the path to the video file in the storage account.
- So the microservice will simply stream video files stored the blob container when `HTTP GET /video?id=<name>` is called. 

In [43]:
!type Flixtube.AzureStorage\Flixtube.AzureStorage\Controllers\AzureStorageController.cs
#!cat Flixtube.AzureStorage/Flixtube.AzureStorage/Controllers/AzureStorageController.cs # use this on Linux/Mac

using System.Net;
using Microsoft.AspNetCore.Mvc;
using Azure.Storage;
using Azure.Storage.Blobs;
using Azure.Storage.Blobs.Models;
// https://learn.microsoft.com/en-us/azure/storage/blobs/storage-quickstart-blobs-dotnet

namespace Flixtube.AzureStorage.Controllers;

[ApiController]
[Route("/")]
public class AzureStorageController : ControllerBase
{
    private readonly ILogger<AzureStorageController> _logger;
    private readonly IConfiguration _config;
    private readonly string STORAGE_ACCOUNT_NAME;
    private readonly string STORAGE_ACCESS_KEY;
    private readonly string STORAGE_CONTAINER_NAME;

    public AzureStorageController(ILogger<AzureStorageController> logger, IConfiguration config)
    {
        _logger = logger;
        _config = config;

        STORAGE_ACCOUNT_NAME = _config.GetValue<string>("STORAGE_ACCOUNT_NAME")!;
        STORAGE_ACCESS_KEY = _config.GetValue<string>("STORAGE_ACCESS_KEY")!;
        STORAGE_CONTAINER_NAME = _config.GetValue<string>("STORAGE_CONTAIN

## Let's look at the Docker file for `Flixtube.AzureStorage`

- The Dockerfile `Flixtube.AzureStorage\Dockerfile` is listed below.
  - It base image has `.net sdk 9.0` pre-installed.
  - It sets the `WORKDIR` to `/src` and copies all code for the microservice to it.
  - Then it restores (installs) all NuGet packages.
  - Finally, it starts the microservice, by issuing the command below when the container starts.
    - `dotnet watch run --no-launch-profile --project Flixtube.AzureStorage.csproj`

In [45]:
!type Flixtube.AzureStorage\Dockerfile
#!cat Flixtube.AzureStorage/Dockerfile # use this on Linux/Mac

FROM mcr.microsoft.com/dotnet/sdk:9.0
WORKDIR /src
EXPOSE 80
COPY ./Flixtube.AzureStorage ./
RUN ["dotnet","restore"]
CMD ["dotnet","watch","run","--no-launch-profile","--project","Flixtube.AzureStorage.csproj"]


## Build and Push a Docker Image to Azure Container Registry

- Here we are building an image of the ASP.NET Web API application (microservice).
- We are tagging the image as `flixtube2025g00.azurecr.io/video-storage:1,`where:
  - `flixtube2025g00.azurecr.io` is the URL (LOGIN SERVER) to our Container Registry.
  - `video-storage` is the name of our image (repository).
  - `1` is the version of the image (tag).
- Then the image is pushed to the Azure Container Registry.
- Finally, the local image is removed from the host computer.

In [47]:
# Build Docker image with Nodejs Application
!docker build -q -t {CONTAINER_REGISTRY_LOGIN_SERVER}/video-storage:1 -f ./Flixtube.AzureStorage/Dockerfile ./Flixtube.AzureStorage

# Push Docker Image to Azure Container Registry
!docker push {CONTAINER_REGISTRY_LOGIN_SERVER}/video-storage:1

# Clean up
!docker rmi {CONTAINER_REGISTRY_LOGIN_SERVER}/video-storage:1
!docker images {CONTAINER_REGISTRY_LOGIN_SERVER}/video-storage:1

sha256:76bcbc1929e4b826e0b2701705c3fde1c0ff47b7d881b4dcc7ce02f3c3fcf251
The push refers to repository [flixtube2025g00.azurecr.io/video-storage]
09a8b39fa149: Preparing
e5d6003d4bc5: Preparing
cb5d5d0673d7: Preparing
1bb564ecf252: Preparing
7e9d6a3f8c92: Preparing
2e7a3a1e4448: Preparing
064dc71a1978: Preparing
c9aaa778af8e: Preparing
5479f1788e98: Preparing
d54764d9a8e7: Preparing
3b245d6409b1: Preparing
f5fe472da253: Preparing
c9aaa778af8e: Waiting
5479f1788e98: Waiting
d54764d9a8e7: Waiting
3b245d6409b1: Waiting
f5fe472da253: Waiting
2e7a3a1e4448: Waiting
064dc71a1978: Waiting
e5d6003d4bc5: Pushed
cb5d5d0673d7: Pushed
1bb564ecf252: Pushed
09a8b39fa149: Pushed
064dc71a1978: Pushed
c9aaa778af8e: Pushed
d54764d9a8e7: Pushed
2e7a3a1e4448: Pushed
5479f1788e98: Pushed
3b245d6409b1: Pushed
f5fe472da253: Pushed
7e9d6a3f8c92: Pushed
1: digest: sha256:dfe2bab0a6cea90ab0ba2109c6fc916bdc40f973bcf6844039f5cb8a5b5ed137 size: 2843
Untagged: flixtube2025g00.azurecr.io/video-storage:1
Untagged: flix

## List Repositories in Container Registry `flixtube2025g00 `

- We see that the Repository `video-storage` has been created in Container Registry `flixtube2025g00`.

In [48]:
!az acr repository list -n flixtube2025g00 --top 10 -o table

Result
-------------
video-storage


## Show Information about Repository `video-storage`

- The Azure CLI command `az acr repository show -n flixtube2025g00 --repository video-storage -o table`
  - Shows information about Repository `video-storage` in Container Registry `flixtube2025g00`.
    - It contains images named `video-storage` (ImageName).
    - It has a tag count of `1` (TagCount), i.e. currently there is only one tag for the `video-storage` image.

In [49]:
!az acr repository show -n flixtube2025g00 --repository video-storage -o table

CreatedTime                   ImageName      LastUpdateTime                ManifestCount    Registry                    TagCount
----------------------------  -------------  ----------------------------  ---------------  --------------------------  ----------
2025-01-28T07:42:41.5857582Z  video-storage  2025-01-28T07:42:41.6694035Z  1                flixtube2025g00.azurecr.io  1


## List Tags in Repository `video-storage`

- The Azure CLI command `az acr repository show-tags -n flixtube2025g00 --repository video-storage --top 10 -o table`:
  - Lists the tags in Repository `video-storage` in Container Registry `flixtube2025g00`.
    - Currently there is only one tag.
    - The tag has the value `1`.

In [50]:
!az acr repository show-tags -n flixtube2025g00 --repository video-storage --top 10 -o table

Result
--------
1


## Show Information about Image `video-streaming:1`

- The Azure CLI command `az acr repository show -n flixtube2025g00 --image video-storage:1 -o table`:
  - Shows information about image `video-storage:1` in Container Registry `flixtube2025g00`.
    - The information includes the Digest for the image.

In [51]:
!az acr repository show -n flixtube2025g00 --image video-storage:1 -o table

CreatedTime                   Digest                                                                   LastUpdateTime                Name    Signed
----------------------------  -----------------------------------------------------------------------  ----------------------------  ------  --------
2025-01-28T07:42:41.6969079Z  sha256:dfe2bab0a6cea90ab0ba2109c6fc916bdc40f973bcf6844039f5cb8a5b5ed137  2025-01-28T07:42:41.6969079Z  1       False


## Show Azure Container Registry Usage

- The command `az acr show-usage -n flixtube2025g00 -o table` shows the usage of Container Registry `flixtube2025g00`.
  - We can see the maximum number of allowed bytes in the `LIMIT` column (first row).
  - We can see the current number of bytes in the `CURRENT VALUE` column (first row).

In [52]:
!az acr show-usage -n flixtube2025g00 -o table

NAME       LIMIT        CURRENT VALUE    UNIT
---------  -----------  ---------------  ------
Size       10737418240  348702922        Bytes
Webhooks   2            0                Count
ScopeMaps  100          0                Count
Tokens     100          0                Count


## Pull and Run an Image from the Azure Container Registry

- The image `video-storage:1` will be pulled from the Azure Container Registry to the host computer.
- Then a Container is created from the Image, where:
  - The host computer's port 3000 is mapped to the container's port 3000.
  - An environment variable `FLIXTUBE_AZURE_STORAGE_PORT` is created in the container with the value `3000`.
  - An environment variable `FLIXTUBE_STORAGE_ACCOUNT_NAME` is created in the container with the Storage Account's Name.
  - An environment variable `FLIXTUBE_STORAGE_ACCESS_KEY` is created in the container with the Storage Access Key.
  - An environment variable `FLIXTUBE_STORAGE_CONTAINER_NAME` is created in the container with the value `videos`.

In [53]:
!docker run --name video-storage -d -p 3000:3000 -e FLIXTUBE_AZURE_STORAGE_PORT=3000 -e FLIXTUBE_STORAGE_ACCOUNT_NAME={STORAGE_ACCOUNT_NAME} -e FLIXTUBE_STORAGE_ACCESS_KEY={STORAGE_ACCESS_KEY} -e FLIXTUBE_STORAGE_CONTAINER_NAME=videos {CONTAINER_REGISTRY_LOGIN_SERVER}/video-storage:1
!docker ps
!docker logs video-storage
#!docker exec -it video-storage /bin/bash

7b55e1907ebd3b11ef9c5d92093f6fd4c1dabe5a250fa4a4792b991a46d7cd60


Unable to find image 'flixtube2025g00.azurecr.io/video-storage:1' locally
1: Pulling from video-storage
486dbf987c66: Already exists
b7f90be4bd50: Already exists
56676eeaaf44: Already exists
b7db1e46071c: Already exists
baddccf2b575: Already exists
3820e4df2c07: Already exists
2da257c5207e: Already exists
e5b6c62eef46: Already exists
44fcc8e6ed98: Already exists
750df5a7f22a: Already exists
65e056afa413: Already exists
e0a8fbc2c7a3: Already exists
Digest: sha256:dfe2bab0a6cea90ab0ba2109c6fc916bdc40f973bcf6844039f5cb8a5b5ed137
Status: Downloaded newer image for flixtube2025g00.azurecr.io/video-storage:1


CONTAINER ID   IMAGE                                        COMMAND                  CREATED                  STATUS                  PORTS                            NAMES
7b55e1907ebd   flixtube2025g00.azurecr.io/video-storage:1   "dotnet watch run --…"   Less than a second ago   Up Less than a second   80/tcp, 0.0.0.0:3000->3000/tcp   video-storage
0fcaba79546f   4a7ae7008ea2                                 "/docker-entrypoint.…"   21 hours ago             Up 21 hours                                              k8s_proxy_kubernetes-dashboard-kong-78fd98d579-g489v_kubernetes-dashboard_d9874745-7e26-4ef6-8eea-1f61c8538df6_0
9fe370cc4473   d9cbc9f4053c                                 "/dashboard-metrics-…"   21 hours ago             Up 21 hours                                              k8s_kubernetes-dashboard-metrics-scraper_kubernetes-dashboard-metrics-scraper-6c8d6bb74d-tv5v5_kubernetes-dashboard_5ed529a7-fa15-47d5-98eb-cbc576ff14fc_0
c3b9888b8dd8   538c5083d89d                 

## Access the Microservice from a Web Browser

- Open a web browser and enter the URL http://localhost:3000/video?id=SampleVideo_1280x720_1mb.mp4
  - This will send an HTTP GET request to the microservice's GET route for the "/video" path.
    - The query string `id=SampleVideo_1280x720_1mb.mp4` will also be accesible by the GET route.
  - The video stored in the storage account's storage container will be streamed to the web browser by the microservice.

In [45]:
#!firefox http://localhost:3000/video?id=SampleVideo_1280x720_1mb.mp4

## Stop and Remove the Docker Container and Image from your computer

- The Container `video-storage` is stopped and removed from the host computer.
- Then the Image `video-storage:1` is removed from the host computer.

In [54]:
!docker stop video-storage
!docker rm video-storage
!docker ps -a
!docker rmi {CONTAINER_REGISTRY_LOGIN_SERVER}/video-storage:1
!docker images {CONTAINER_REGISTRY_LOGIN_SERVER}/video-storage:1

video-storage
video-storage
CONTAINER ID   IMAGE          COMMAND                  CREATED        STATUS        PORTS     NAMES
0fcaba79546f   4a7ae7008ea2   "/docker-entrypoint.…"   21 hours ago   Up 21 hours             k8s_proxy_kubernetes-dashboard-kong-78fd98d579-g489v_kubernetes-dashboard_d9874745-7e26-4ef6-8eea-1f61c8538df6_0
9fe370cc4473   d9cbc9f4053c   "/dashboard-metrics-…"   21 hours ago   Up 21 hours             k8s_kubernetes-dashboard-metrics-scraper_kubernetes-dashboard-metrics-scraper-6c8d6bb74d-tv5v5_kubernetes-dashboard_5ed529a7-fa15-47d5-98eb-cbc576ff14fc_0
c3b9888b8dd8   538c5083d89d   "/dashboard-auth"        21 hours ago   Up 21 hours             k8s_kubernetes-dashboard-auth_kubernetes-dashboard-auth-f7d869bcb-66r6j_kubernetes-dashboard_aef1e7ca-0388-407b-83f1-16196a54642f_0
834831d96cf5   71e2af47d086   "/dashboard-api --in…"   21 hours ago   Up 21 hours             k8s_kubernetes-dashboard-api_kubernetes-dashboard-api-c9c479bb4-w56zq_kubernetes-dashboard_64a17

## Logout from the Azure Container Registry via Docker

In [55]:
!docker logout $CONTAINER_REGISTRY_LOGIN_SERVER

Removing login credentials for flixtube2025g00.azurecr.io


## Delete Repository `video-storage` from Azure Container Registry

- The Azure CLI command `az acr repository delete -y -n flixtube2025g00 --repository video-storage -o table`
  - Deletes the Repository `video-storage` in Container Registry `flixtube2025g00`.
  - Use this command to delete all images in Repository `video-storage`.
- The Azure CLI command `az acr repository delete -y -n flixtube2025g00 --image video-storage:1 -table`
  - Deletes the Image `video-storage:1` in Container Registry `flixtube2025g00`.
  - Use this command to delete one image from Repository `video-storage`.

In [56]:
#!az acr repository delete -y -n flixtube2025g00 --image video-storage:1 -table
!az acr repository delete -y -n flixtube2025g00 --repository video-storage -o table

## List Repositories in Container Registry `flixtube2025g00 `

- We see that the Repository `video-storage` has been deleted from Container Registry `flixtube2025g00`.

In [57]:
!az acr repository list -n flixtube2025g00 --top 10 -o table

## Terraform Destroy

- We destroy the Terraform Project as before.

- Once again it's better to run this command in a separate terminal.
  - Open a new terminal.
  - Make sure you are in the folder `workshop5/01_Azure_and_Terraform/05_storage_account`
  - Execute the command `terraform destroy -auto-approve`

The output should look something like the below ...

```bash
azurerm_resource_group.main: Refreshing state... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00]
azurerm_container_registry.main: Refreshing state... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00]
azurerm_storage_account.main: Refreshing state... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.Storage/storageAccounts/flixtube2025g00]
azurerm_storage_container.main: Refreshing state... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.Storage/storageAccounts/flixtube2025g00/blobServices/default/containers/videos]

Terraform used the selected providers to generate the following execution plan. Resource actions are indicated with the following symbols:
  - destroy

Terraform will perform the following actions:

  # azurerm_container_registry.main will be destroyed
  - resource "azurerm_container_registry" "main" {
      - admin_enabled                 = true -> null
      - admin_password                = (sensitive value) -> null
      - admin_username                = "flixtube2025g00" -> null
      - anonymous_pull_enabled        = false -> null
      - data_endpoint_enabled         = false -> null
      - encryption                    = [] -> null
      - export_policy_enabled         = true -> null
      - id                            = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00" -> null
      - location                      = "westeurope" -> null
      - login_server                  = "flixtube2025g00.azurecr.io" -> null
      - name                          = "flixtube2025g00" -> null
      - network_rule_bypass_option    = "AzureServices" -> null
      - network_rule_set              = [] -> null
      - public_network_access_enabled = true -> null
      - quarantine_policy_enabled     = false -> null
      - resource_group_name           = "flixtube2025g00" -> null
      - retention_policy_in_days      = 0 -> null
      - sku                           = "Basic" -> null
      - tags                          = {} -> null
      - trust_policy_enabled          = false -> null
      - zone_redundancy_enabled       = false -> null
    }

  # azurerm_resource_group.main will be destroyed
  - resource "azurerm_resource_group" "main" {
      - id         = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00" -> null
      - location   = "westeurope" -> null
      - name       = "flixtube2025g00" -> null
      - tags       = {} -> null
        # (1 unchanged attribute hidden)
    }

  # azurerm_storage_account.main will be destroyed
  - resource "azurerm_storage_account" "main" {
      - access_tier                        = "Hot" -> null
      - account_kind                       = "StorageV2" -> null
      - account_replication_type           = "LRS" -> null
      - account_tier                       = "Standard" -> null
      - allow_nested_items_to_be_public    = true -> null
      - cross_tenant_replication_enabled   = false -> null
      - default_to_oauth_authentication    = false -> null
      - dns_endpoint_type                  = "Standard" -> null
      - https_traffic_only_enabled         = true -> null
      - id                                 = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.Storage/storageAccounts/flixtube2025g00" -> null
      - infrastructure_encryption_enabled  = false -> null
      - is_hns_enabled                     = false -> null
      - large_file_share_enabled           = false -> null
      - local_user_enabled                 = true -> null
      - location                           = "westeurope" -> null
      - min_tls_version                    = "TLS1_2" -> null
      - name                               = "flixtube2025g00" -> null
      - nfsv3_enabled                      = false -> null
      - primary_access_key                 = (sensitive value) -> null
      - primary_blob_connection_string     = (sensitive value) -> null
      - primary_blob_endpoint              = "https://flixtube2025g00.blob.core.windows.net/" -> null
      - primary_blob_host                  = "flixtube2025g00.blob.core.windows.net" -> null
      - primary_connection_string          = (sensitive value) -> null
      - primary_dfs_endpoint               = "https://flixtube2025g00.dfs.core.windows.net/" -> null
      - primary_dfs_host                   = "flixtube2025g00.dfs.core.windows.net" -> null
      - primary_file_endpoint              = "https://flixtube2025g00.file.core.windows.net/" -> null
      - primary_file_host                  = "flixtube2025g00.file.core.windows.net" -> null
      - primary_location                   = "westeurope" -> null
      - primary_queue_endpoint             = "https://flixtube2025g00.queue.core.windows.net/" -> null
      - primary_queue_host                 = "flixtube2025g00.queue.core.windows.net" -> null
      - primary_table_endpoint             = "https://flixtube2025g00.table.core.windows.net/" -> null
      - primary_table_host                 = "flixtube2025g00.table.core.windows.net" -> null
      - primary_web_endpoint               = "https://flixtube2025g00.z6.web.core.windows.net/" -> null
      - primary_web_host                   = "flixtube2025g00.z6.web.core.windows.net" -> null
      - public_network_access_enabled      = true -> null
      - queue_encryption_key_type          = "Service" -> null
      - resource_group_name                = "flixtube2025g00" -> null
      - secondary_access_key               = (sensitive value) -> null
      - secondary_connection_string        = (sensitive value) -> null
      - sftp_enabled                       = false -> null
      - shared_access_key_enabled          = true -> null
      - table_encryption_key_type          = "Service" -> null
      - tags                               = {} -> null
        # (56 unchanged attributes hidden)

      - blob_properties {
          - change_feed_enabled           = false -> null
          - change_feed_retention_in_days = 0 -> null
          - last_access_time_enabled      = false -> null
          - versioning_enabled            = false -> null
            # (1 unchanged attribute hidden)
        }

      - queue_properties {
          - hour_metrics {
              - enabled               = true -> null
              - include_apis          = true -> null
              - retention_policy_days = 7 -> null
              - version               = "1.0" -> null
            }
          - logging {
              - delete                = false -> null
              - read                  = false -> null
              - retention_policy_days = 0 -> null
              - version               = "1.0" -> null
              - write                 = false -> null
            }
          - minute_metrics {
              - enabled               = false -> null
              - include_apis          = false -> null
              - retention_policy_days = 0 -> null
              - version               = "1.0" -> null
            }
        }

      - share_properties {
          - retention_policy {
              - days = 7 -> null
            }
        }
    }

  # azurerm_storage_container.main will be destroyed
  - resource "azurerm_storage_container" "main" {
      - container_access_type             = "private" -> null
      - default_encryption_scope          = "$account-encryption-key" -> null
      - encryption_scope_override_enabled = true -> null
      - has_immutability_policy           = false -> null
      - has_legal_hold                    = false -> null
      - id                                = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.Storage/storageAccounts/flixtube2025g00/blobServices/default/containers/videos" -> null
      - metadata                          = {} -> null
      - name                              = "videos" -> null
      - resource_manager_id               = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.Storage/storageAccounts/flixtube2025g00/blobServices/default/containers/videos" -> null
      - storage_account_id                = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.Storage/storageAccounts/flixtube2025g00" -> null
    }

Plan: 0 to add, 0 to change, 4 to destroy.

Changes to Outputs:
  - AZURE_CONTAINER_REGISTRY_HOSTNAME = "flixtube2025g00.azurecr.io" -> null
  - AZURE_CONTAINER_REGISTRY_PASSWORD = (sensitive value) -> null
  - AZURE_CONTAINER_REGISTRY_USERNAME = "flixtube2025g00" -> null
  - AZURE_STORAGE_ACCOUNT_KEY         = (sensitive value) -> null
  - AZURE_STORAGE_ACCOUNT_NAME        = "flixtube2025g00" -> null
azurerm_storage_container.main: Destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.Storage/storageAccounts/flixtube2025g00/blobServices/default/containers/videos]
azurerm_container_registry.main: Destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00]
azurerm_storage_container.main: Destruction complete after 0s
azurerm_storage_account.main: Destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.Storage/storageAccounts/flixtube2025g00]
azurerm_storage_account.main: Destruction complete after 4s
azurerm_container_registry.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...nerRegistry/registries/flixtube2025g00, 10s elapsed]   
azurerm_container_registry.main: Destruction complete after 17s
azurerm_resource_group.main: Destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00]
azurerm_resource_group.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00, 10s elapsed]       
azurerm_resource_group.main: Destruction complete after 17s

Destroy complete! Resources: 4 destroyed.
```

In [59]:
# !terraform destroy -auto-approve

## List Azure Container Registries

- The Azure CLI command `az acr list -o table` lists all Container Registries in Azure.
- We see that the Container Registry defined in the Terraform project has been destroyed.

In [60]:
!az acr list -o table

## List Azure Storage Accounts

- The Azure CLI command `az storage account list -o table` lists all Storage Accounts in Azure.
- We see that the Storage Account defined in the Terraform project has been destroyed.

In [61]:
#!az storage account list --resource-group $RESOURCE_GROUP -o table
!az storage account list -o table

## List Azure Resource Groups

- The Azure CLI command `az group list -o table` lists all Resource Groups in Azure.
- We see that the Resource Group defined in the Terraform project has been destroyed.

In [62]:
!az group list -o table